# IMPORTS

general install for using open ai models

In [336]:
import openai
from openai import OpenAI
import json
import io
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_distances

specific libraries for making images machine readable for chatgpt vision

In [337]:
from PIL import Image
from io import BytesIO
import base64
import re
import os
import csv

# CREDENTIALS AND AUTHENTICATIONS

In [ ]:
#client = OpenAI()

# FUNCTIONS

In [339]:
import os
import re
import shutil

# Define the directories
train_dir = '/home/salvador_cb/3_term/Applied_research_studio/data/img/train'
overlap_dir = os.path.join(train_dir, 'overlap')
elevation_dir = os.path.join(train_dir, 'elevation')
street_dir = os.path.join(train_dir, 'street')
elevation_2_dir = os.path.join(train_dir, 'elevation_2')
street_2_dir = os.path.join(train_dir, 'street_2')

# Create the new directories if they don't exist
os.makedirs(elevation_2_dir, exist_ok=True)
os.makedirs(street_2_dir, exist_ok=True)

# Get the list of files in the overlap directory
overlap_files = os.listdir(overlap_dir)

# Extract city names from overlap file names using regex
city_names = []
for filename in overlap_files:
    match = re.match(r'^(.*)_overlap\.png$', filename)
    if match:
        city_names.append(match.group(1))  # Extract the city name

# Copy files from elevation and street folders to the new directories
for city_name in city_names:
    # Define the file paths
    elevation_file = os.path.join(elevation_dir, f"{city_name}_elevation.png")
    street_file = os.path.join(street_dir, f"{city_name}_street_network.png")

    # Copy elevation file to elevation_2 if it exists
    if os.path.exists(elevation_file):
        shutil.copy(elevation_file, elevation_2_dir)

    # Copy street file to street_2 if it exists
    if os.path.exists(street_file):
        shutil.copy(street_file, street_2_dir)

# Verify the number of files in the new directories
elevation_2_files = os.listdir(elevation_2_dir)
street_2_files = os.listdir(street_2_dir)

print(f"Number of files in elevation_2: {len(elevation_2_files)}")
print(f"Number of files in street_2: {len(street_2_files)}")
print(f"Number of files in overlap: {len(overlap_files)}")

Number of files in elevation_2: 1365
Number of files in street_2: 1365
Number of files in overlap: 1365


Turning an image to its b64 encoding (-> sending it as text)

In [340]:
def image_to_base64(image):
    buffered = BytesIO()
    image.save(buffered, format="PNG")  # You can change the format to PNG or other types if needed
    img_str = base64.b64encode(buffered.getvalue()).decode('utf-8')
    return img_str

Function for chatgpt vision

In [341]:
def gpt_vision(vision_prompt, image_encoded):
  response = client.chat.completions.create(
      model="chatgpt-4o-latest",
      messages=[
          {
            "role": "user",
            "content": [
              {"type": "text", "text": vision_prompt},
              { "type": "image_url",
                "image_url": {
                  "url": "data:image/jpeg;base64,"+ image_encoded,
                },
              },
            ],
          },
        ],
        max_tokens=5000,
  )
  return response.choices[0].message.content

In [342]:
def gptVisionApp(prompt, image):
  # encode to b64
  image_encoded = image_to_base64(image)

  global gptVisionAppPromptTemplate
  final_prompt = gptVisionAppPromptTemplate + prompt

  res = gpt_vision(final_prompt,image_encoded)

  #optionally collect results in a list (defined in the cell below)
  #anwser_collection.append(res)
  return res


Regex functions

In [343]:
def city_name_regex(img_path):
    match = re.search(r'/home/salvador_cb/3_term/Applied_research_studio/data/img/train/overlap/(.*)_overlap\.png', img_path)
    if match:
        city_country = match.group(1)
        # Split the city and country by the last comma
        parts = city_country.rsplit(", ", 1)
        if len(parts) == 2:
            city, country = parts
            return city.strip(), country.strip()
    # Return None for both city and country if the regex does not match
    return None, None

In [344]:
def tags_regex(description):
    match = re.search(r"\[(.*?)\]", description)
    if match:
        tags_list = match.group(1).replace('"', '').split(',')
        return tags_list
    return []

In [345]:
def desc_regex(img_desc):
    return re.sub(r'\[.*?\]', '', img_desc).strip()

# MAIN

In [362]:
csv_path = '/home/salvador_cb/3_term/Applied_research_studio/data/csv/overlap_metadata.csv'
if os.path.exists(csv_path):
    overlap_df = pd.read_csv(csv_path)
else:
    overlap_df = pd.DataFrame(columns=["country", "city", "overlap_file_name", "overlap_description", "overlap_tags"])

processed_cities = list(overlap_df['overlap_file_name'])

folder_path = '/home/salvador_cb/3_term/Applied_research_studio/data/img/train/overlap'

city_counter = 0
# Loop through all the files in the folder
for filename in os.listdir(folder_path):
    if filename.endswith('.png'):  # Assuming all your images are in PNG format
        # Construct the full file path
        file_path = os.path.join(folder_path, filename)
        city, country = city_name_regex(file_path)
        city_counter += 1
        print(f"Processing {city_counter} of {len(os.listdir(folder_path))} images: {filename}")

        if filename in processed_cities:
            print(f"{filename} is already processed. Skipping...")
            continue
        
        print(f'city: {city}, country: {country}')
        image_vis = Image.open(file_path)
        base64_image = image_to_base64(image_vis)
        
        # Use regex to extract the city and country names from the filename
        global gptVisionAppPromptTemplate
        gptVisionAppPromptTemplate = f"Here is a street layout map of the city {city}, {country}. give me a description and tags of the urban characteristics."

        img_desc = gpt_vision(f"""You are a model with advanced vision capabilities.
            Your task is to analyze the provided image and generate a detailed description of its features. The image shows an overlay of an urban street network on a grayscale topographic map of {city}, {country}. In this image, red lines represent streets and pathways, while the grayscale background illustrates the terrain, where lighter areas indicate higher elevations and darker areas indicate lower regions.
            Provide a comprehensive description of how the urban street network interacts with the underlying topography. Highlight how the street layout adapts to the terrain's features. For example, describe if a grid structure is situated in a valley, if roads curve to navigate around mountain ridges, if dense intersections are located near flat areas, or if arterial roads traverse elevated regions. Use specific observations to explain this interaction, such as noting a dense grid pattern in flat, low-elevation zones or winding roads along steep, elevated contours.
            Start with a general overview of how the urban network integrates with the natural landscape, then provide detail on specific areas of interest. Avoid using proper names or landmarks; focus solely on the physical characteristics and spatial relationships between the urban elements and the terrain. The description must be at least 200 characters.

            Additionally, return a Python list with exactly 10 tags or keywords that summarize the observed urban and topographic features. The format should be: ["tag1", "tag2", "tag3", "tag4", "tag5", "tag6", "tag7", "tag8", "tag9", "tag10"]. Use only descriptive terms relevant to the image, such as valley, mountain, grid, branching, winding, elevation, intersection, hillside, and contour.
            Do not include any markdown elements in your response, just the one body description and the list of tags.
            Example Output:
            "The image depicts a dense urban street network concentrated in a central valley, surrounded by elevated terrain. The streets in the valley form a grid-like pattern, while roads on the outskirts curve and meander to adapt to the hilly topography. A prominent arterial road cuts through the valley, connecting the central grid to the surrounding areas. The elevated regions feature sparse, winding roads that follow the contours of the terrain, with intersections primarily located at flatter areas near the valley edges. The overall layout demonstrates a clear adaptation of the urban network to the natural landscape, with dense development in the lowlands and sparse connectivity in the highlands."
            ["valley", "grid", "elevation", "winding", "intersection", "arterial road", "hillside", "contour", "dense", "sparse"]
            """, base64_image)
        tags = tags_regex(img_desc)
        tags_str = ','.join(tags)
        desc = desc_regex(img_desc)
        
        new_row = {'country': country, 'city': city, 'overlap_file_name': filename, 'overlap_description': desc, 'overlap_tags': tags_str}
        overlap_df = pd.concat([overlap_df, pd.DataFrame([new_row])], ignore_index=True)

Processing 1 of 1365 images: Xiamen, China_overlap.png
Xiamen, China_overlap.png is already processed. Skipping...
Processing 2 of 1365 images: Drammen, Norway_overlap.png
Drammen, Norway_overlap.png is already processed. Skipping...
Processing 3 of 1365 images: Reggio Emilia, Italy_overlap.png
Reggio Emilia, Italy_overlap.png is already processed. Skipping...
Processing 4 of 1365 images: Cologne, Germany_overlap.png
Cologne, Germany_overlap.png is already processed. Skipping...
Processing 5 of 1365 images: Yaounde, Cameroon_overlap.png
Yaounde, Cameroon_overlap.png is already processed. Skipping...
Processing 6 of 1365 images: Punta del Este, Uruguay_overlap.png
Punta del Este, Uruguay_overlap.png is already processed. Skipping...
Processing 7 of 1365 images: Constantine, Algeria_overlap.png
Constantine, Algeria_overlap.png is already processed. Skipping...
Processing 8 of 1365 images: Santiago de Chile, Chile_overlap.png
Santiago de Chile, Chile_overlap.png is already processed. Skip

In [363]:
overlap_df

,country,city,overlap_file_name,overlap_description,overlap_tags
0,China,Xiamen,"Xiamen, China_overlap.png",The image depicts an urban street network that...,"valley, grid, elevation, winding, intersection..."
1,Norway,Drammen,"Drammen, Norway_overlap.png",The image depicts a well-defined urban street ...,"valley, grid, elevation, winding, intersection..."
2,Italy,Reggio Emilia,"Reggio Emilia, Italy_overlap.png",The image illustrates a striking contrast betw...,"valley, grid, elevation, winding, intersection..."
3,Germany,Cologne,"Cologne, Germany_overlap.png",The image shows an urban street network densel...,"valley, grid, elevation, winding, intersection..."
4,Cameroon,Yaounde,"Yaounde, Cameroon_overlap.png",The image shows a densely woven urban street n...,"valley, grid, elevation, winding, intersection..."
...,...,...,...,...,...
1360,South Korea,Ulsan,"Ulsan, South Korea_overlap.png",The image illustrates an urban street network ...,"valley, grid, elevation, winding, intersection..."
1361,Ireland,Kilkenny,"Kilkenny, Ireland_overlap.png",The image depicts an urban street network over...,"valley, grid, elevation, winding, intersection..."
1362,Austria,Wels,"Wels, Austria_overlap.png",The image shows a dense urban street network p...,"valley, grid, elevation, winding, intersection..."
1363,Hong Kong,Sha Tin,"Sha Tin, Hong Kong_overlap.png",The image displays an urban street network lay...,"valley, grid, elevation, winding, intersection..."


In [364]:
overlap_df.to_csv('/home/salvador_cb/3_term/Applied_research_studio/data/csv/overlap_metadata.csv', index=False, quoting=csv.QUOTE_ALL)

# streets

In [294]:
csv_path = '/home/salvador_cb/3_term/Applied_research_studio/data/csv/street_metadata.csv'
if os.path.exists(csv_path):
    street_df = pd.read_csv(csv_path)
else:
    street_df = pd.DataFrame(columns=["country", "city", "street_file_name", "street_description", "street_tags"])

processed_cities = list(street_df['street_file_name'])

folder_path = '/home/salvador_cb/3_term/Applied_research_studio/data/img/train/street_2'

city_counter = 0
# Loop through all the files in the folder
for filename in os.listdir(folder_path):
    if filename.endswith('.png'):  # Assuming all your images are in PNG format
        # Construct the full file path
        file_path = os.path.join(folder_path, filename)
        city, country = city_name_regex(file_path)
        city_counter += 1
        print(f"Processing {city_counter} of {len(os.listdir(folder_path))} images: {filename}")

        if filename in processed_cities:
            print(f"{filename} is already processed. Skipping...")
            continue
        
        print(f'city: {city}, country: {country}')
        image_vis = Image.open(file_path)
        base64_image = image_to_base64(image_vis)
        
        # Use regex to extract the city and country names from the filename
        global gptVisionAppPromptTemplate
        gptVisionAppPromptTemplate = f"Here is a street layout map of the city {city}, {country}. give me a description and tags of the urban characteristics."

        img_desc = gpt_vision(f"""You are a model with vision capabilities.
            Your task is to generate a description of the images.
            Each image shows the street network of {city}, {country}.
            The streets are represented as red lines on a white background. The density, layout, and structure of the streets vary across the image.
            Return a text description of the image, describing the urban features of the street network. Focus on aspects such as the density of the streets, the shape and layout of the network (e.g., grid-like, radial, irregular), and the location of the most prominent elements (e.g., intersections, highways, main roads, clusters of streets). For example: "The image shows a dense grid-like street network in the center, with a radial pattern of highways extending outward. The main intersections are located in the top-left and bottom-right corners."
            Start the description from the more general aspects (e.g., "The image shows a dense urban area with a grid-like street network") and then describe the more detailed elements. Avoid using the names of places or streets in the description; focus only on the characteristics of the street network.
            The description should be at least 200 characters.
            Also, return a python list with tags or keywords that describe the image. There should be 10 tags. The format is this ["tag1","tag2","tag3","tag4","tag5","tag6","tag7","tag8","tag9","tag10"].
            Use tags that describe urban features present in the description, such as "grid", "radial", "dense", "sparse", "intersection", "highway", "main road", "cluster", "irregular", "periphery".
            Do not include any markdown elements in your response, just the description and the list of tags.""", base64_image)
        tags = tags_regex(img_desc)
        tags_str = ','.join(tags)
        desc = desc_regex(img_desc)
        
        new_row = {'country': country, 'city': city, 'street_file_name': filename, 'street_description': desc, 'street_tags': tags_str}
        street_df = pd.concat([street_df, pd.DataFrame([new_row])], ignore_index=True)

Processing 1 of 1365 images: Surat, India_street_network.png
city: Surat, country: India
Processing 2 of 1365 images: Moscow, Russia_street_network.png
city: Moscow, country: Russia
Processing 3 of 1365 images: Rome, Italy_street_network.png
city: Rome, country: Italy
Processing 4 of 1365 images: Quito, Ecuador_street_network.png
city: Quito, country: Ecuador
Processing 5 of 1365 images: Kalamata, Greece_street_network.png
city: Kalamata, country: Greece
Processing 6 of 1365 images: Francistown, Botswana_street_network.png
city: Francistown, country: Botswana
Processing 7 of 1365 images: Dharan, Nepal_street_network.png
city: Dharan, country: Nepal
Processing 8 of 1365 images: Differdange, Luxembourg_street_network.png
city: Differdange, country: Luxembourg
Processing 9 of 1365 images: Monrovia, Liberia_street_network.png
city: Monrovia, country: Liberia
Processing 10 of 1365 images: Mazatenango, Guatemala_street_network.png
city: Mazatenango, country: Guatemala
Processing 11 of 1365 i

In [295]:
street_df

,country,city,street_file_name,street_description,street_tags
0,India,Surat,"Surat, India_street_network.png",The image shows a dense urban area with a high...,"grid, dense, irregular, intersection, main roa..."
1,Russia,Moscow,"Moscow, Russia_street_network.png",The image shows a dense urban area at the cent...,"radial,dense,intersection,main road,highway,cl..."
2,Italy,Rome,"Rome, Italy_street_network.png",The image shows a dense urban area with an irr...,"dense, irregular, radial, main road, highway, ..."
3,Ecuador,Quito,"Quito, Ecuador_street_network.png",The image shows a dense urban area with a mix ...,"dense,grid,irregular,main road,intersection,cl..."
4,Greece,Kalamata,"Kalamata, Greece_street_network.png",The image shows a dense urban area near the so...,"dense,grid,irregular,main road,intersection,cl..."
...,...,...,...,...,...
1360,Oman,Nizwa,"Nizwa, Oman_street_network.png",The image shows a moderately dense urban area ...,"dense,grid,irregular,radial,main road,intersec..."
1361,Zimbabwe,Marondera,"Marondera, Zimbabwe_street_network.png",The image shows a moderately dense urban area ...,"grid,irregular,dense,cluster,main road,interse..."
1362,Brazil,Campinas,"Campinas, Brazil_street_network.png",The image shows a dense urban area with a high...,"grid, dense, radial, main road, intersection, ..."
1363,Guatemala,Mixco,"Mixco, Guatemala_street_network.png",The image shows a highly varied urban area wit...,"grid, dense, irregular, sparse, intersection, ..."


In [296]:
street_df.to_csv('/home/salvador_cb/3_term/Applied_research_studio/data/csv/street_metadata.csv', index=False, quoting=csv.QUOTE_ALL)

In [245]:
df = pd.DataFrame(columns=["country","city","description","tags"])

In [203]:
df

,country,city,description,tags
0,Lebanon,Beirut,The image shows a coastal region bordered by a...,"coast,ocean,mountain,hill,elevation,slope,upla..."
1,Pakistan,Peshawar,The image displays a generally flat to gently ...,"plain,hill,river,stream,lowland,drainage,eleva..."
2,Denmark,Roskilde,The image displays a coastal terrain with a mi...,"coast,bay,river,estuary,island,peninsula,uplan..."
3,Guyana,Linden,The image shows a region dominated by low-lyin...,"river,plain,valley,tributary,basin,stream,hill..."
4,Azerbaijan,Ganja,The image depicts a landscape dominated by an ...,"valley,river,tributary,ridge,plateau,floodplai..."
...,...,...,...,...
1139,Kazakhstan,Shymkent,The image shows a rugged mountainous region wi...,"mountain,valley,ridge,river,channel,ravine,gul..."
1140,Benin,Porto-Novo,The image presents a flat coastal region with ...,"river,plains,estuary,coast,tributary,wetland,d..."
1141,Bulgaria,Pleven,The image depicts a landscape characterized by...,"valley,hill,ridge,slope,river,watercourse,eros..."
1142,Botswana,Maun,The image displays a gently varied landscape w...,"highland,plain,valley,river,floodplain,elevati..."


In [ ]:
existing_df.to_csv(csv_path, index=False, quoting=csv.QUOTE_ALL)

In [ ]:
img_desc = gpt_vision(f"""You are a model with vision capabilities.
Your task is to generate a description of the images.
Each image shows an overlay of an urban street network on a grayscale topographic map of {city_name_regex(file_path)[0]}, {city_name_regex(file_path)[1]}. In the image, red lines represent streets and pathways, while the grayscale background depicts the terrain. Whiter areas indicate higher elevations and darker areas indicate lower regions.
Return a detailed text description of the image that explains how the urban street network interacts with the underlying topography. Describe how the street layout adapts to the terrain's features. For instance, if a grid structure is nested within a valley, if roads bend to circumnavigate mountain ridges, if a dense node of intersections is located near a coastal margin, or if arterial roads cut through elevated areas. Use specific observations to illustrate this interplay, such as noting a straight, dense grid pattern in flat, low-elevation zones and winding, meandering roads along sharper, elevated contours.
Begin with a general overview of how the urban network is integrated with the natural landscape, then transition into more detailed commentary on particular areas of interest. Do not include any proper names or landmarks; focus solely on describing the physical characteristics and spatial relationships between the urban elements and the terrain. The description must be at least 200 characters.
Additionally, return a Python list with 10 tags or keywords that encapsulate the observed urban and topographic features. The format should be: ["tag1","tag2","tag3","tag4","tag5","tag6","tag7","tag8","tag9","tag10"]. Use only specific descriptive terms related to the image, such as valley, mountain, coast, grid, branching, winding, elevation, intersection, hillside, and contour.
Do not include any markdown formatting in your response; provide only the description and the list of tags.
""", base64_image)

 The image resembles a sample where a dense cluster of red lines in the center branches outward, suggesting a complex, interconnected network that adapts to the natural contours of the land.

In [249]:
df = pd.read_csv('/home/salvador_cb/3_term/Applied_research_studio/data/csv/new_metadata.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1365 entries, 0 to 1364
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   country      1365 non-null   object
 1   city         1365 non-null   object
 2   description  1365 non-null   object
 3   tags         1365 non-null   object
dtypes: object(4)
memory usage: 42.8+ KB


In [365]:
csv_path = '/home/salvador_cb/3_term/Applied_research_studio/data/csv/new_metadata.csv'
if os.path.exists(csv_path):
    existing_df = pd.read_csv(csv_path)
else:
    existing_df = pd.DataFrame(columns=["country", "city", "elevation_file_name", "description", "tags"])

processed_cities = list(zip(existing_df['city'], existing_df['country']))

folder_path = '/home/salvador_cb/3_term/Applied_research_studio/data/img/train/elevation_2'

city_counter = 0
# Loop through all the files in the folder
for filename in os.listdir(folder_path):
    if filename.endswith('.png'):  # Assuming all your images are in PNG format
        # Construct the full file path
        file_path = os.path.join(folder_path, filename)
        city, country = city_name_regex(file_path)
        city_counter += 1
        print(f"Processing {city_counter} of {len(os.listdir(folder_path))} images: {filename}")

        if (city, country) in processed_cities:
            print(f"City {city}, {country} is already processed. Skipping...")
            continue
        
        print(f'city: {city}, country: {country}')
        image_vis = Image.open(file_path)
        base64_image = image_to_base64(image_vis)
        
        # Use regex to extract the city and country names from the filename
        global gptVisionAppPromptTemplate
        gptVisionAppPromptTemplate = f"Here is a topograpical map of the city {city}, {country}. give me a description and tags of the topographical characteristics."

        img_desc = gpt_vision(f"""You are a model with vision capabilities.
            Your task is to generate a description of the images.
            each image shows the topography of {city}, {country}.
            The altitude is represented with a grayscale color gradient, where the whitest areas are the tallest and the darkest areas are at the bottom. the whiter the pixel the higher the point is.
            Return a text description of the image, describing the topographical features (rivers, ocean, mountains, valleys, plains, etc...). describe where the elements are located in the image, and their relative size and shape. For example: 3 mountains in the top and a river crossing the image from left to right. The mountains are large and slopped, while the river is thin and winding.
            Start the description from the more general aspects (the image shows an island, the image shows a valley, etc ...) and then describe the more detailed elements. Use the name of the place better identify what the topographical elements are (if its an ocean or a lake). Do not use name of places in the description, such as the city name or name of a river, only the characteristics of the terrain. The description be at least 200 characters.
            also, return a python list with tags or keywords that describe the image. there should be 10 tags. the format is this ["tag1","tag2","tag3","tag4","tag5","tag6","tag7","tag8","tag9","tag10"].
            Do not use tags such as topography or elevation. Use only topographical features present in the description, such as ocean, mountain, river, lake, valley, island, coast, etc...
            Do not include any markdown elements in your response, just the description and the list of tags.""", base64_image)
        tags = tags_regex(img_desc)
        tags_str = ','.join(tags)
        desc = desc_regex(img_desc)
        
        new_row = {'country': country, 'city': city, 'elevation_file_name': filename}
        existing_df = pd.concat([existing_df, pd.DataFrame([new_row])], ignore_index=True)

Processing 1 of 1365 images: Beirut, Lebanon_elevation.png
city: None, country: None
Processing 2 of 1365 images: Peshawar, Pakistan_elevation.png
city: None, country: None
Processing 3 of 1365 images: Roskilde, Denmark_elevation.png
city: None, country: None
Processing 4 of 1365 images: Linden, Guyana_elevation.png
city: None, country: None


KeyboardInterrupt: 

In [366]:
existing_df

,country,city,description,tags,elevation_file_name
0,Lebanon,Beirut,The image shows a coastal region bordered by a...,"coast,ocean,mountain,hill,elevation,slope,upla...",NaN
1,Pakistan,Peshawar,The image displays a generally flat to gently ...,"plain,hill,river,stream,lowland,drainage,eleva...",NaN
2,Denmark,Roskilde,The image displays a coastal terrain with a mi...,"coast,bay,river,estuary,island,peninsula,uplan...",NaN
3,Guyana,Linden,The image shows a region dominated by low-lyin...,"river,plain,valley,tributary,basin,stream,hill...",NaN
4,Azerbaijan,Ganja,The image depicts a landscape dominated by an ...,"valley,river,tributary,ridge,plateau,floodplai...",NaN
...,...,...,...,...,...
1363,UK,Leeds,The image depicts a rugged and hilly landscape...,"valley,river,ridge,hill,ravine,channel,highlan...",NaN
1364,United Kingdom,London,The image shows a varied inland terrain charac...,"valley,river,upland,highlands,hill,tributary,r...",NaN
1365,None,None,NaN,NaN,"Beirut, Lebanon_elevation.png"
1366,None,None,NaN,NaN,"Peshawar, Pakistan_elevation.png"


In [ ]:
existing_df.to_csv(csv_path, index=False, quoting=csv.QUOTE_ALL)

In [204]:
df.to_csv('/home/salvador_cb/3_term/Applied_research_studio/data/csv/new_metadata.csv', index=False, quoting=csv.QUOTE_ALL)

In [370]:
import pandas as pd
import os

# Path to the CSV file
csv_path = '/home/salvador_cb/3_term/Applied_research_studio/data/csv/new_metadata.csv'

# Load the existing CSV
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
else:
    raise FileNotFoundError(f"{csv_path} does not exist.")

# Ensure the column is added after 'city'
if 'elevation_file_name' not in df.columns:
    df.insert(df.columns.get_loc('city') + 1, 'elevation_file_name', '')

# Define the folder containing elevation files
elevation_folder = '/home/salvador_cb/3_term/Applied_research_studio/data/img/train/elevation_2'

# Loop through the rows and populate the 'elevation_file_name' column
for index, row in df.iterrows():
    city = row['city']
    country = row['country']
    elevation_file = f"{city}, {country}_elevation.png"
    print(elevation_file)
    elevation_file_path = os.path.join(elevation_folder, elevation_file)
    
    # Check if the file exists
    if os.path.exists(elevation_file_path):
        df.at[index, 'elevation_file_name'] = elevation_file
    else:
        df.at[index, 'elevation_file_name'] = None  # Set to None if the file doesn't exist

# Save the updated DataFrame back to the CSV

Beirut, Lebanon_elevation.png
Peshawar, Pakistan_elevation.png
Roskilde, Denmark_elevation.png
Linden, Guyana_elevation.png
Ganja, Azerbaijan_elevation.png
Ettelbruck, Luxembourg_elevation.png
Amsterdam, Netherlands_elevation.png
Male, Maldives_elevation.png
Schaan, Liechtenstein_elevation.png
Kutaisi, Georgia_elevation.png
Lugano, Switzerland_elevation.png
Taipei, Taiwan_elevation.png
Kohtla-Jarve, Estonia_elevation.png
Charleroi, Belgium_elevation.png
Hobart, Australia_elevation.png
Maturin, Venezuela_elevation.png
Villavicencio, Colombia_elevation.png
Montreal, Canada_elevation.png
Sucre, Bolivia_elevation.png
Bucharest, Romania_elevation.png
Tsuen Wan, Hong Kong_elevation.png
Heredia, Costa Rica_elevation.png
Belfast, United Kingdom_elevation.png
Liverpool, United Kingdom_elevation.png
Serowe, Botswana_elevation.png
La Serena, Chile_elevation.png
Osijek, Croatia_elevation.png
Turin, Italy_elevation.png
Kaga-Bandoro, Central African Republic_elevation.png
Kaduna, Nigeria_elevation.p

In [371]:
df

,country,city,elevation_file_name,description,tags
0,Lebanon,Beirut,"Beirut, Lebanon_elevation.png",The image shows a coastal region bordered by a...,"coast,ocean,mountain,hill,elevation,slope,upla..."
1,Pakistan,Peshawar,"Peshawar, Pakistan_elevation.png",The image displays a generally flat to gently ...,"plain,hill,river,stream,lowland,drainage,eleva..."
2,Denmark,Roskilde,"Roskilde, Denmark_elevation.png",The image displays a coastal terrain with a mi...,"coast,bay,river,estuary,island,peninsula,uplan..."
3,Guyana,Linden,"Linden, Guyana_elevation.png",The image shows a region dominated by low-lyin...,"river,plain,valley,tributary,basin,stream,hill..."
4,Azerbaijan,Ganja,"Ganja, Azerbaijan_elevation.png",The image depicts a landscape dominated by an ...,"valley,river,tributary,ridge,plateau,floodplai..."
...,...,...,...,...,...
1360,Australia,Newcastle,"Newcastle, Australia_elevation.png",The image shows a coastal area with a prominen...,"coast,river,ocean,inlet,estuary,floodplain,hil..."
1361,Canada,Hamilton,"Hamilton, Canada_elevation.png",The image shows a coastal region with a signif...,"coast,lake,hill,plateau,valley,river,stream,es..."
1362,Panama,Santiago,"Santiago, Panama_elevation.png",The image shows a mountainous inland region wi...,"mountain,valley,peak,ridge,canyon,highlands,st..."
1363,UK,Leeds,"Leeds, UK_elevation.png",The image depicts a rugged and hilly landscape...,"valley,river,ridge,hill,ravine,channel,highlan..."


In [372]:
df.to_csv(csv_path, index=False, quoting=csv.QUOTE_ALL)

In [373]:
import pandas as pd

# Load the datasets
new_metadata_path = '/home/salvador_cb/3_term/Applied_research_studio/data/csv/new_metadata.csv'
street_metadata_path = '/home/salvador_cb/3_term/Applied_research_studio/data/csv/street_metadata.csv'
overlap_metadata_path = '/home/salvador_cb/3_term/Applied_research_studio/data/csv/overlap_metadata.csv'

new_metadata = pd.read_csv(new_metadata_path)
street_metadata = pd.read_csv(street_metadata_path)
overlap_metadata = pd.read_csv(overlap_metadata_path)

# Merge the datasets on 'country' and 'city'
merged_df = pd.merge(new_metadata, street_metadata, on=['country', 'city'], how='inner')
merged_df = pd.merge(merged_df, overlap_metadata, on=['country', 'city'], how='inner')

# Reorder columns to match the desired structure
merged_df = merged_df[[
    "country", "city",
    "elevation_file_name", "elevation_description", "elevation_tags",
    "street_file_name", "street_description", "street_tags",
    "overlap_file_name", "overlap_description", "overlap_tags"
]]


In [374]:
merged_df

,country,city,elevation_file_name,elevation_description,elevation_tags,street_file_name,street_description,street_tags,overlap_file_name,overlap_description,overlap_tags
0,Lebanon,Beirut,"Beirut, Lebanon_elevation.png",The image shows a coastal region bordered by a...,"coast,ocean,mountain,hill,elevation,slope,upla...","Beirut, Lebanon_street_network.png",The image shows a largely dense urban area wit...,"dense, irregular, grid, main road, intersectio...","Beirut, Lebanon_overlap.png",The image depicts a varied urban street networ...,"grid, valley, elevation, winding, hillside, ar..."
1,Pakistan,Peshawar,"Peshawar, Pakistan_elevation.png",The image displays a generally flat to gently ...,"plain,hill,river,stream,lowland,drainage,eleva...","Peshawar, Pakistan_street_network.png",The image shows a highly varied street network...,"dense,grid,irregular,intersection,main road,hi...","Peshawar, Pakistan_overlap.png",The image presents an urban street network sup...,"valley, grid, elevation, winding, intersection..."
2,Denmark,Roskilde,"Roskilde, Denmark_elevation.png",The image displays a coastal terrain with a mi...,"coast,bay,river,estuary,island,peninsula,uplan...","Roskilde, Denmark_street_network.png",The image shows a central dense urban area wit...,"dense,irregular,grid,radial,main road,intersec...","Roskilde, Denmark_overlap.png",The image shows an urban street network intric...,"valley, grid, elevation, winding, intersection..."
3,Guyana,Linden,"Linden, Guyana_elevation.png",The image shows a region dominated by low-lyin...,"river,plain,valley,tributary,basin,stream,hill...","Linden, Guyana_street_network.png",The image shows a moderately dense urban stree...,"irregular,grid,cluster,intersection,main road,...","Linden, Guyana_overlap.png",The image shows a red urban street network ove...,"valley, grid, elevation, winding, intersection..."
4,Azerbaijan,Ganja,"Ganja, Azerbaijan_elevation.png",The image depicts a landscape dominated by an ...,"valley,river,tributary,ridge,plateau,floodplai...","Ganja, Azerbaijan_street_network.png",The image shows a dense urban area with a comb...,"grid,irregular,dense,sparse,main road,intersec...","Ganja, Azerbaijan_overlap.png",The image reveals a concentrated urban street ...,"valley, grid, elevation, winding, intersection..."
...,...,...,...,...,...,...,...,...,...,...,...
1360,Australia,Newcastle,"Newcastle, Australia_elevation.png",The image shows a coastal area with a prominen...,"coast,river,ocean,inlet,estuary,floodplain,hil...","Newcastle, Australia_street_network.png",The image shows a moderately dense urban area ...,"grid,irregular,dense,sparse,intersection,main ...","Newcastle, Australia_overlap.png",The image reveals an urban street network laid...,"valley, grid, elevation, winding, intersection..."
1361,Canada,Hamilton,"Hamilton, Canada_elevation.png",The image shows a coastal region with a signif...,"coast,lake,hill,plateau,valley,river,stream,es...","Hamilton, Canada_street_network.png",The image shows a varied urban area with a com...,"grid,dense,sparse,main road,intersection,clust...","Hamilton, Canada_overlap.png",The image depicts a striking interaction betwe...,"valley, grid, elevation, winding, intersection..."
1362,Panama,Santiago,"Santiago, Panama_elevation.png",The image shows a mountainous inland region wi...,"mountain,valley,peak,ridge,canyon,highlands,st...","Santiago, Panama_street_network.png",The image shows a moderately dense urban area ...,"grid,dense,irregular,main road,intersection,cl...","Santiago, Panama_overlap.png",The image shows a dense urban street network o...,"valley, grid, elevation, winding, intersection..."
1363,UK,Leeds,"Leeds, UK_elevation.png",The image depicts a rugged and hilly landscape...,"valley,river,ridge,hill,ravine,channel,highlan...","Leeds, UK_street_network.png",The image shows a densely built-up urban core ...,"dense,irregular,grid,radial,intersection,clust...","Leeds, UK_overlap.png",The image displays a complex urban street netw

In [379]:
# Save the merged dataset to a new CSV file
output_path = '/home/salvador_cb/3_term/Applied_research_studio/data/csv/merged_metadata.csv'
merged_df.to_csv(output_path, index=False, quoting=csv.QUOTE_ALL)

print(f"Merged dataset saved to {output_path}")

Merged dataset saved to /home/salvador_cb/3_term/Applied_research_studio/data/csv/merged_metadata.csv


In [388]:
import pandas as pd
import csv

merged_df = pd.read_csv('/home/salvador_cb/3_term/Applied_research_studio/data/csv/merged_metadata.csv')

# Update file paths to include folder structure
merged_df['elevation_file_name'] = 'elevation/' + merged_df['elevation_file_name']
merged_df['street_file_name'] = 'street/' + merged_df['street_file_name']
merged_df['overlap_file_name'] = 'overlap/' + merged_df['overlap_file_name']

# Reorder columns to match the desired structure
merged_df = merged_df[[
    "country", "city",
    "elevation_file_name", "elevation_description", "elevation_tags",
    "street_file_name", "street_description", "street_tags",
    "overlap_file_name", "overlap_description", "overlap_tags"
]]

In [389]:
merged_df

,country,city,elevation_file_name,elevation_description,elevation_tags,street_file_name,street_description,street_tags,overlap_file_name,overlap_description,overlap_tags
0,Lebanon,Beirut,"elevation/Beirut, Lebanon_elevation.png",The image shows a coastal region bordered by a...,"coast,ocean,mountain,hill,elevation,slope,upla...","street/Beirut, Lebanon_street_network.png",The image shows a largely dense urban area wit...,"dense, irregular, grid, main road, intersectio...","overlap/Beirut, Lebanon_overlap.png",The image depicts a varied urban street networ...,"grid, valley, elevation, winding, hillside, ar..."
1,Pakistan,Peshawar,"elevation/Peshawar, Pakistan_elevation.png",The image displays a generally flat to gently ...,"plain,hill,river,stream,lowland,drainage,eleva...","street/Peshawar, Pakistan_street_network.png",The image shows a highly varied street network...,"dense,grid,irregular,intersection,main road,hi...","overlap/Peshawar, Pakistan_overlap.png",The image presents an urban street network sup...,"valley, grid, elevation, winding, intersection..."
2,Denmark,Roskilde,"elevation/Roskilde, Denmark_elevation.png",The image displays a coastal terrain with a mi...,"coast,bay,river,estuary,island,peninsula,uplan...","street/Roskilde, Denmark_street_network.png",The image shows a central dense urban area wit...,"dense,irregular,grid,radial,main road,intersec...","overlap/Roskilde, Denmark_overlap.png",The image shows an urban street network intric...,"valley, grid, elevation, winding, intersection..."
3,Guyana,Linden,"elevation/Linden, Guyana_elevation.png",The image shows a region dominated by low-lyin...,"river,plain,valley,tributary,basin,stream,hill...","street/Linden, Guyana_street_network.png",The image shows a moderately dense urban stree...,"irregular,grid,cluster,intersection,main road,...","overlap/Linden, Guyana_overlap.png",The image shows a red urban street network ove...,"valley, grid, elevation, winding, intersection..."
4,Azerbaijan,Ganja,"elevation/Ganja, Azerbaijan_elevation.png",The image depicts a landscape dominated by an ...,"valley,river,tributary,ridge,plateau,floodplai...","street/Ganja, Azerbaijan_street_network.png",The image shows a dense urban area with a comb...,"grid,irregular,dense,sparse,main road,intersec...","overlap/Ganja, Azerbaijan_overlap.png",The image reveals a concentrated urban street ...,"valley, grid, elevation, winding, intersection..."
...,...,...,...,...,...,...,...,...,...,...,...
1360,Australia,Newcastle,"elevation/Newcastle, Australia_elevation.png",The image shows a coastal area with a prominen...,"coast,river,ocean,inlet,estuary,floodplain,hil...","street/Newcastle, Australia_street_network.png",The image shows a moderately dense urban area ...,"grid,irregular,dense,sparse,intersection,main ...","overlap/Newcastle, Australia_overlap.png",The image reveals an urban street network laid...,"valley, grid, elevation, winding, intersection..."
1361,Canada,Hamilton,"elevation/Hamilton, Canada_elevation.png",The image shows a coastal region with a signif...,"coast,lake,hill,plateau,valley,river,stream,es...","street/Hamilton, Canada_street_network.png",The image shows a varied urban area with a com...,"grid,dense,sparse,main road,intersection,clust...","overlap/Hamilton, Canada_overlap.png",The image depicts a striking interaction betwe...,"valley, grid, elevation, winding, intersection..."
1362,Panama,Santiago,"elevation/Santiago, Panama_elevation.png",The image shows a mountainous inland region wi...,"mountain,valley,peak,ridge,canyon,highlands,st...","street/Santiago, Panama_street_network.png",The image shows a moderately dense urban area ...,"grid,dense,irregular,main road,intersection,cl...","overlap/Santiago, Panama_overlap.png",The image shows a dense urban street network o...,"valley, grid, elevation, winding, intersection..."
1363,UK,Leeds,"elevation/Leeds, UK_elevation.png",The image depicts a rugged and hilly landscape...,"valley,river,ridge,hill,ravine,channel,highl

In [390]:
# Save the merged dataset to a new CSV file
output_path = '/home/salvador_cb/3_term/Applied_research_studio/data/csv/new_merged_metadata.csv'
merged_df.to_csv(output_path, index=False, quoting=csv.QUOTE_ALL)

print(f"Merged dataset saved to {output_path}")

Merged dataset saved to /home/salvador_cb/3_term/Applied_research_studio/data/csv/new_merged_metadata.csv
